In [1]:
import os, glob, time, requests, io, ssl, certifi, urllib.request
import pandas as pd, numpy as np
from datetime import datetime, timedelta
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from math import erf, sqrt

# -------------------------------------------------
# 1) Bring in your ESPN functions from the other notebook
#    (expects pull_players_and_teams_for_date to exist)
# -------------------------------------------------
%run ESPNDATA.ipynb
# --- Silence prints from ESPNDATA.ipynb helpers ---
from contextlib import redirect_stdout
import io

def pull_players_and_teams_for_date_quiet(ds):
    buf = io.StringIO()
    with redirect_stdout(buf):                 # swallow ESPNDATA prints
        return pull_players_and_teams_for_date(ds)  # original function


# ---------- scalar-safe normalizer used everywhere ----------
def norm_key(x):
    """Normalize team names consistently to a lowercase key."""
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

# -------------------------------------------------
# 2) Utilities
# -------------------------------------------------
def payoff_from_american(odds):
    """Return decimal payout per $1 stake (EV calc uses this).
       Accepts float/str/NaN; returns NaN if missing."""
    if pd.isna(odds):
        return np.nan
    o = int(round(float(odds)))
    return o/100.0 if o > 0 else 100.0/abs(o)

def normal_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def p_over_from_mu_sigma(line, mu, sigma):
    if sigma <= 1e-6:
        return float(mu > line)
    z = (line - mu) / sigma
    return 1 - normal_cdf(z)

# -------------------------------------------------
# 3) Ensure team points exist in teams_df (via ESPN summary header)
# -------------------------------------------------
SITE_SUMMARY = "https://site.web.api.espn.com/apis/site/v2/sports/football/nfl/summary"

def _j(url):
    r = requests.get(url, timeout=20); r.raise_for_status(); return r.json()

def _scores_for_event(event_id: str):
    h = _j(f"{SITE_SUMMARY}?event={event_id}").get("header", {})
    comp = (h.get("competitions") or [{}])[0]
    out = []
    for c in comp.get("competitors", []):
        team_name = c.get("team", {}).get("displayName")
        pts = c.get("score")
        if pts is None:
            ls = c.get("linescores") or []
            if ls:
                pts = sum(int(x.get("value") or 0) for x in ls)
        if team_name is not None and pts is not None:
            out.append({"event_id": str(event_id), "team": team_name, "points": int(pts)})
    return out

def ensure_points(teams_df: pd.DataFrame) -> pd.DataFrame:
    df = teams_df.copy()
    for candidate in ("points","pts","score"):
        if candidate in df.columns:
            return df.rename(columns={candidate: "points"}) if candidate != "points" else df

    score_rows = []
    for eid in df["event_id"].astype(str).unique():
        try:
            score_rows.extend(_scores_for_event(eid))
        except Exception:
            continue
        time.sleep(0.06)

    scores = pd.DataFrame(score_rows).dropna(subset=["points"])
    if scores.empty:
        raise ValueError("Couldn't derive team points; inspect teams_df and a sample summary JSON.")

    out = df.merge(scores, on=["event_id","team"], how="left")

    if out["points"].isna().any():
        df2 = out[out["points"].isna()].copy()
        ok  = out[out["points"].notna()]
        if not df2.empty:
            df2["team_key"] = df2["team"].map(norm_key)
            scores["team_key"] = scores["team"].map(norm_key)
            df2 = df2.drop(columns=["points"]).merge(
                scores.drop(columns=["team"]).drop_duplicates(["event_id","team_key"]),
                on=["event_id","team_key"], how="left"
            ).drop(columns=["team_key"])
            out = pd.concat([ok, df2], ignore_index=True)
    return out

# -------------------------------------------------
# 4) Build training set from ESPN team totals
# -------------------------------------------------
def pull_team_totals_for_dates(date_list):
    rows = []
    for ds in date_list:
        _, teams_df = pull_players_and_teams_for_date_quiet(ds)
        if teams_df.empty:
            continue
        rows.append(teams_df.assign(asof_date=ds))
        time.sleep(0.08)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def build_game_table_from_teams(teams_df):
    t = ensure_points(teams_df.copy())
    t['team_key'] = t['team'].map(norm_key)

    g = t[['event_id','team','team_key','points','asof_date']]
    g_sorted = g.sort_values(['event_id','team_key'])
    pairs = []
    for eid, grp in g_sorted.groupby('event_id'):
        if len(grp) != 2:
            continue
        a, b = grp.iloc[0], grp.iloc[1]
        pairs.append({
            'event_id': eid,
            'team_a': a.team, 'team_b': b.team,
            'team_a_key': a.team_key, 'team_b_key': b.team_key,
            'pts_a': pd.to_numeric(a.points, errors='coerce'),
            'pts_b': pd.to_numeric(b.points, errors='coerce'),
            'asof_date': a.asof_date
        })
    games = pd.DataFrame(pairs).dropna(subset=['pts_a','pts_b'])
    games['total_points'] = games['pts_a'] + games['pts_b']
    return games

def rolling_team_features(games, window=3):
    a = games[['event_id','asof_date','team_a_key','pts_a','pts_b']].rename(
        columns={'team_a_key':'team_key','pts_a':'pts_for','pts_b':'pts_against'})
    b = games[['event_id','asof_date','team_b_key','pts_b','pts_a']].rename(
        columns={'team_b_key':'team_key','pts_b':'pts_for','pts_a':'pts_against'})
    long = pd.concat([a,b], ignore_index=True).sort_values(['team_key','asof_date','event_id'])

    feats = []
    for team, grp in long.groupby('team_key'):
        grp = grp.copy()
        grp['pf_l3'] = grp['pts_for'].shift(1).rolling(window).mean()
        grp['pa_l3'] = grp['pts_against'].shift(1).rolling(window).mean()
        grp['pf_l5'] = grp['pts_for'].shift(1).rolling(5).mean()
        grp['pa_l5'] = grp['pts_against'].shift(1).rolling(5).mean()
        feats.append(grp.assign(team_key=team))
    f = pd.concat(feats, ignore_index=True)

    fa = f[['event_id','team_key','pf_l3','pa_l3','pf_l5','pa_l5']]
    fb = fa.copy()
    games2 = games.merge(fa, left_on=['event_id','team_a_key'], right_on=['event_id','team_key'], how='left') \
                  .drop(columns=['team_key']) \
                  .rename(columns={'pf_l3':'a_pf_l3','pa_l3':'a_pa_l3','pf_l5':'a_pf_l5','pa_l5':'a_pa_l5'})
    games2 = games2.merge(fb, left_on=['event_id','team_b_key'], right_on=['event_id','team_key'], how='left') \
                   .drop(columns=['team_key']) \
                   .rename(columns={'pf_l3':'b_pf_l3','pa_l3':'b_pa_l3','pf_l5':'b_pf_l5','pa_l5':'b_pa_l5'})
    return games2

def train_total_model(train_games):
    features = ['a_pf_l3','a_pa_l3','a_pf_l5','a_pa_l5','b_pf_l3','b_pa_l3','b_pf_l5','b_pa_l5']
    X = train_games[features].fillna(train_games[features].mean())
    y = train_games['total_points']
    model = Ridge(alpha=3.0).fit(X, y)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    preds, ys = [], []
    for tr, te in kf.split(X):
        m = Ridge(alpha=3.0).fit(X.iloc[tr], y.iloc[tr])
        p = m.predict(X.iloc[te])
        preds.append(p); ys.append(y.iloc[te].values)
    resid = np.concatenate(ys) - np.concatenate(preds)
    sigma = np.std(resid, ddof=1)
    return model, sigma, features

def build_training_from_dates(n_days=200):
    today = datetime.utcnow().date()
    dates = [(today - timedelta(days=i)).strftime("%Y%m%d") for i in range(n_days)]
    ttot = pull_team_totals_for_dates(dates)
    games = build_game_table_from_teams(ttot)
    games = rolling_team_features(games)
    games = games.dropna(subset=['a_pf_l3','b_pf_l3'])
    return games

# -------------------------------------------------
# 5) READ GAME TOTALS (Over/Under) FROM GOOGLE SHEETS
# -------------------------------------------------
SHEET_CSV_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQtfhqFKMwFDldCgWJp4Lb5wqm71F2EXUdwYD_75VxAMPlyUsoMaWct5KrYwXJyPScMxTKLjonLrEbB/pub?gid=0&single=true&output=csv"

def _pick_col(df, candidates):
    cmap = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in df.columns: return c
        if c.lower() in cmap: return cmap[c.lower()]
    return None

def read_totals_from_google_sheet(csv_url: str) -> pd.DataFrame:
    """Return one row per game+book with book, home_team_api, away_team_api, point,
       commence_time, price_over, price_under, home_key, away_key.
       Supports 'wide' (price_over/price_under) or 'long' (Over/Under rows) layouts."""
    # Robust CSV fetch (handles SSL constraints)
    try:
        ctx = ssl.create_default_context(cafile=certifi.where())
        with urllib.request.urlopen(csv_url, context=ctx, timeout=30) as resp:
            raw = pd.read_csv(io.BytesIO(resp.read()))
    except Exception:
        r = requests.get(csv_url, timeout=30)
        r.raise_for_status()
        raw = pd.read_csv(io.BytesIO(r.content))

    home_col = _pick_col(raw, ["home_team_api","home_team","home"])
    away_col = _pick_col(raw, ["away_team_api","away_team","away"])
    book_col = _pick_col(raw, ["book","bookmaker"])
    point_col = _pick_col(raw, ["point","total","line","total_points"])
    time_col  = _pick_col(raw, ["commence_time","start_time","kickoff"])
    market_col= _pick_col(raw, ["market"])
    name_col  = _pick_col(raw, ["name","label"])              # 'Over'/'Under' in long format
    price_col = _pick_col(raw, ["price","odds","american_odds"])
    over_col  = _pick_col(raw, ["price_over","over_price","overodds","over"])
    under_col = _pick_col(raw, ["price_under","under_price","underodds","under"])

    if home_col is None or away_col is None or book_col is None:
        raise ValueError(f"Missing core columns (home/away/book). Found: {list(raw.columns)}")

    df = raw.copy()
    # Filter to totals only when the sheet actually labels it as totals
    if market_col and market_col in df.columns:
        mvals = df[market_col].astype(str).str.lower()
        if (mvals == "totals").any():
            df = df[mvals == "totals"].copy()

    # Wide format
    if over_col and under_col and point_col:
        use_cols = [book_col, home_col, away_col, point_col, over_col, under_col] + ([time_col] if time_col else [])
        w = df[use_cols].rename(columns={
            book_col: "book",
            home_col: "home_team_api",
            away_col: "away_team_api",
            point_col: "point",
            (time_col or "commence_time"): "commence_time",
            over_col: "price_over",
            under_col: "price_under",
        })
    else:
        # Long format: join Over/Under rows
        if not (name_col and price_col and point_col):
            raise ValueError("Sheet isn’t a totals layout. Need a 'point' (line) and Over/Under prices.")
        on_cols = [book_col, home_col, away_col, point_col] + ([time_col] if time_col else [])
        over  = df[df[name_col].astype(str).str.lower().eq("over") ][on_cols + [price_col]].copy()
        under = df[df[name_col].astype(str).str.lower().eq("under")][on_cols + [price_col]].copy()
        m = over.merge(under, on=on_cols, suffixes=("_over","_under")).rename(columns={
            book_col: "book",
            home_col: "home_team_api",
            away_col: "away_team_api",
            point_col: "point",
            (time_col or "commence_time"): "commence_time",
            f"{price_col}_over":  "price_over",
            f"{price_col}_under": "price_under",
        })
        w = m

    # Clean types & keys
    w["price_over"]  = pd.to_numeric(w.get("price_over"),  errors="coerce")
    w["price_under"] = pd.to_numeric(w.get("price_under"), errors="coerce")
    w["point"]       = pd.to_numeric(w.get("point"),       errors="coerce")
    if "commence_time" not in w.columns:
        w["commence_time"] = pd.NaT

    w["home_key"] = w["home_team_api"].map(norm_key)
    w["away_key"] = w["away_team_api"].map(norm_key)
    return w

# =================================================
# ===== 1) TRAIN
# =================================================
train_games = build_training_from_dates(n_days=200)
model, sigma, FEATURES = train_total_model(train_games)
print("Training games:", train_games.shape, "| Estimated sigma:", round(float(sigma), 2))

# =================================================
# ===== 2) READ ODDS FROM GOOGLE SHEETS
# =================================================
odds = read_totals_from_google_sheet(SHEET_CSV_URL)
if odds.empty:
    raise RuntimeError("Odds sheet returned 0 rows. Double-check the sheet/tab & column names.")

# =================================================
# ===== 3) SCORE VS ODDS 
# =================================================
latest = train_games.copy()

def get_latest_feats(team_key, side_prefix='a'):
    cols = [f'{side_prefix}_pf_l3', f'{side_prefix}_pa_l3', f'{side_prefix}_pf_l5', f'{side_prefix}_pa_l5']
    a = latest[latest['team_a_key']==team_key][['event_id','team_a_key']+cols].tail(1)
    if not a.empty:
        return a.iloc[0][cols].values
    cols_b = ['b_pf_l3','b_pa_l3','b_pf_l5','b_pa_l5']
    b = latest[latest['team_b_key']==team_key][['event_id','team_b_key']+cols_b].tail(1)
    if not b.empty:
        return b.iloc[0][cols_b].values
    return [np.nan, np.nan, np.nan, np.nan]

# Build feature rows per upcoming game/book from the sheet
pred_rows = []
for _, r in odds.iterrows():
    home_k = norm_key(r['home_team_api'])
    away_k = norm_key(r['away_team_api'])
    a_feats = get_latest_feats(home_k, 'a')
    b_feats = get_latest_feats(away_k, 'b')
    pred_rows.append({
        'home_key': home_k, 'away_key': away_k,
        'book': r['book'], 'total_line': r['point'],
        'price_over': r['price_over'], 'price_under': r['price_under'],
        'commence_time': r['commence_time'],
        'a_pf_l3': a_feats[0], 'a_pa_l3': a_feats[1], 'a_pf_l5': a_feats[2], 'a_pa_l5': a_feats[3],
        'b_pf_l3': b_feats[0], 'b_pa_l3': b_feats[1], 'b_pf_l5': b_feats[2], 'b_pa_l5': b_feats[3],
    })

pred_df = pd.DataFrame(pred_rows)

# Drop rows missing critical inputs for scoring
pred_df = pred_df.dropna(subset=["total_line", "price_over", "price_under"])

# Predict totals and uncertainty
X_new = pred_df[FEATURES].fillna(train_games[FEATURES].mean())
pred_df['pred_total_mu'] = model.predict(X_new)
pred_df['pred_sigma'] = sigma

# Probabilities from Normal model
def p_over_from_mu_sigma(line, mu, sigma):
    if sigma <= 1e-6:
        return float(mu > line)
    z = (line - mu) / sigma
    from math import erf, sqrt
    cdf = 0.5 * (1.0 + erf(z / sqrt(2.0)))
    return 1 - cdf

pred_df['p_over']  = pred_df.apply(lambda r: p_over_from_mu_sigma(r['total_line'], r['pred_total_mu'], r['pred_sigma']), axis=1)
pred_df['p_under'] = 1 - pred_df['p_over']

# EVs
def payoff_from_american(odds):
    if pd.isna(odds): return np.nan
    o = int(round(float(odds)))
    return o/100.0 if o > 0 else 100.0/abs(o)

pred_df['payoff_over']  = pred_df['price_over'].map(payoff_from_american)
pred_df['payoff_under'] = pred_df['price_under'].map(payoff_from_american)

pred_df['ev_over']  = pred_df['p_over']  * pred_df['payoff_over']  - (1 - pred_df['p_over'])
pred_df['ev_under'] = pred_df['p_under'] * pred_df['payoff_under'] - (1 - pred_df['p_under'])

# Pick side, edge, confidence
pred_df['side']       = np.where(pred_df['ev_over'] >= pred_df['ev_under'], 'Over', 'Under')
pred_df['edge']       = pred_df[['ev_over','ev_under']].max(axis=1)
pred_df['confidence'] = np.where(pred_df['side']=='Over', pred_df['p_over'], pred_df['p_under'])

# Kelly sizing (quarter Kelly)
def kelly_fraction(p, odds_american, q=0.25):
    if pd.isna(p) or pd.isna(odds_american): return 0.0
    b = payoff_from_american(odds_american)
    if pd.isna(b) or b <= 0: return 0.0
    f = (p*(b+1)-1)/b
    return max(0.0, q*float(f))

pred_df['kelly_frac'] = np.where(
    pred_df['side']=='Over',
    pred_df.apply(lambda r: kelly_fraction(r['p_over'],  r['price_over']),  axis=1),
    pred_df.apply(lambda r: kelly_fraction(r['p_under'], r['price_under']), axis=1)
)

# Columns to show
cols = ['commence_time','book','home_key','away_key','total_line',
        'pred_total_mu','pred_sigma','side','confidence','edge','kelly_frac',
        'price_over','price_under','p_over','p_under']

# Build the sorted recommendations table
recos = pred_df[cols].sort_values('edge', ascending=False).reset_index(drop=True)

# --- Pretty, labeled table of the top N picks (with headers) ---
def show_top_picks(recos_df, top_n=20):
    if recos_df is None or recos_df.empty:
        print("No recommendations to show.")
        return

    cols_pretty = [
        "commence_time","book","home_key","away_key","total_line",
        "pred_total_mu","side","confidence","edge","kelly_frac",
        "price_over","price_under","p_over","p_under"
    ]
    rename_map = {
        "commence_time":"Date/Time",
        "book":"Book",
        "home_key":"Home",
        "away_key":"Away",
        "total_line":"O/U Line",
        "pred_total_mu":"Model Total",
        "side":"Pick",
        "confidence":"Conf",
        "edge":"Edge($/1)",
        "kelly_frac":"Kelly f",
        "price_over":"Over odds",
        "price_under":"Under odds",
        "p_over":"P(Over)",
        "p_under":"P(Under)"
    }

    t = recos_df.loc[:, cols_pretty].rename(columns=rename_map).copy()
    for c in ["O/U Line","Model Total","Conf","Edge($/1)","Kelly f",
              "Over odds","Under odds","P(Over)","P(Under)"]:
        if c in t.columns:
            t[c] = pd.to_numeric(t[c], errors="coerce").round(3)

    pd.set_option("display.width", 180)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_colwidth", 40)

    print("\n=== TOP PICKS (by edge) ===")
    print(t.head(top_n).to_string(index=False, header=True))

# Show the top 20 in a clear, labeled table
show_top_picks(recos, top_n=20)

# Optional: also save to CSV for your app
recos.to_csv("ou_recommendations_upcoming.csv", index=False)




Found 16 games via: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?dates=2024&seasontype=2&week=1


KeyboardInterrupt: 

KeyboardInterrupt: 

In [15]:
# Pretty, labeled table of the top N picks (with headers)
def show_top_picks(recos_df, top_n=20):
    if recos_df is None or recos_df.empty:
        print("No recommendations to show.")
        return

    # Select & rename columns to readable labels
    cols = [
        "commence_time","book","home_key","away_key","total_line",
        "pred_total_mu","side","confidence","edge","kelly_frac",
        "price_over","price_under","p_over","p_under"
    ]
    rename_map = {
        "commence_time":"Date/Time",
        "book":"Book",
        "home_key":"Home",
        "away_key":"Away",
        "total_line":"O/U Line",
        "pred_total_mu":"Model Total",
        "side":"Pick",
        "confidence":"Conf",
        "edge":"Edge($/1)",
        "kelly_frac":"Kelly f",
        "price_over":"Over odds",
        "price_under":"Under odds",
        "p_over":"P(Over)",
        "p_under":"P(Under)"
    }

    t = recos_df.loc[:, cols].rename(columns=rename_map).copy()

    # Round numeric columns for readability
    for c in ["O/U Line","Model Total","Conf","Edge($/1)","Kelly f","Over odds","Under odds","P(Over)","P(Under)"]:
        if c in t.columns:
            t[c] = pd.to_numeric(t[c], errors="coerce").round(3)

    # Wider print so headers don’t disappear
    pd.set_option("display.width", 180)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_colwidth", 40)

    print("\n=== TOP PICKS (by edge) ===")
    print(t.head(top_n).to_string(index=False, header=True))

# Run it
show_top_picks(recos, top_n=50)



=== TOP PICKS (by edge) ===
 Date/Time         Book                 Home                  Away  O/U Line  Model Total Pick  Conf  Edge($/1)  Kelly f  Over odds  Under odds  P(Over)  P(Under)
10/26/2025 BetOnline.ag       denver broncos        dallas cowboys       2.5       59.160 Over 1.000      3.010     0.25      301.0        1.69    1.000     0.000
10/26/2025 BetOnline.ag       denver broncos        dallas cowboys       2.5       59.160 Over 1.000      3.010     0.25      301.0        2.31    1.000     0.000
10/26/2025 BetOnline.ag       denver broncos        dallas cowboys       2.5       59.160 Over 1.000      3.010     0.25      301.0        2.01    1.000     0.000
10/26/2025 BetOnline.ag       denver broncos        dallas cowboys       2.5       59.160 Over 1.000      3.010     0.25      301.0        1.69    1.000     0.000
10/26/2025 BetOnline.ag       denver broncos        dallas cowboys       2.5       59.160 Over 1.000      3.010     0.25      301.0        2.31    1.000    

In [14]:
# =========================
# 📊 CLEAR DIAGNOSTICS + PICKS SUMMARY (no 'squared' kwarg)
# =========================
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_model_cv(train_df, feature_cols, k=5, alpha=3.0, seed=42):
    X = train_df[feature_cols].copy().fillna(train_df[feature_cols].mean())
    y = train_df["total_points"].astype(float).values

    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    preds, actuals = [], []
    for tr, te in kf.split(X):
        m = Ridge(alpha=alpha).fit(X.iloc[tr], y[tr])
        p = m.predict(X.iloc[te])
        preds.append(p.astype(float))
        actuals.append(y[te].astype(float))

    p_all = np.concatenate(preds)
    y_all = np.concatenate(actuals)

    mse  = mean_squared_error(y_all, p_all)           # no 'squared' kwarg
    rmse = float(np.sqrt(mse))
    mae  = float(mean_absolute_error(y_all, p_all))
    r2   = float(r2_score(y_all, p_all))
    ae   = np.abs(y_all - p_all)

    return {
        "CV folds": k,
        "RMSE (pts)": round(rmse, 2),
        "MAE (pts)": round(mae, 2),
        "R^2": round(r2, 3),
        "Abs Err p50": round(float(np.percentile(ae, 50)), 2),
        "Abs Err p75": round(float(np.percentile(ae, 75)), 2),
        "Abs Err p90": round(float(np.percentile(ae, 90)), 2),
    }

def pretty_print_dict(d, title=None):
    if title:
        print("\n" + "="*len(title))
        print(title)
        print("="*len(title))
    for k, v in d.items():
        print(f"{k:>14}: {v}")

def print_upcoming_summary(pred_df, recos, top_n=20):
    if pred_df is None or len(pred_df) == 0:
        print("No upcoming predictions available.")
        return

    pos_ev   = (pred_df[["ev_over","ev_under"]].max(axis=1) > 0).sum()
    n_rows   = len(pred_df)
    over_ct  = int((pred_df["side"] == "Over").sum())
    under_ct = int((pred_df["side"] == "Under").sum())
    avg_edge = float(pred_df[["ev_over","ev_under"]].max(axis=1).mean())
    conf_vec = np.where(pred_df["side"].eq("Over"), pred_df["p_over"], pred_df["p_under"])
    avg_conf = float(np.nanmean(conf_vec))

    pretty_print_dict(
        {
            "Books x games": n_rows,
            "Positive-EV rows": int(pos_ev),
            "Avg edge ($/1)": round(avg_edge, 3),
            "Avg confidence": round(avg_conf, 3),
            "Over picks": over_ct,
            "Under picks": under_ct,
            "Sigma used": round(float(pred_df.get("pred_sigma", pd.Series([np.nan])).iloc[0]), 2),
        },
        title="UPCOMING PICKS — SUMMARY"
    )

    buckets = pd.cut(conf_vec, bins=[0.5, 0.55, 0.60, 0.65, 1.01],
                     labels=["50–55%", "55–60%", "60–65%", "65%+"])
    bucket_tbl = buckets.value_counts().sort_index()
    print("\nConfidence buckets (count of picks):")
    for idx, val in bucket_tbl.items():
        print(f"  {idx:>6}: {int(val)}")

    cols = ["commence_time","book","home_key","away_key","total_line",
            "pred_total_mu","side","confidence","edge","kelly_frac",
            "price_over","price_under","p_over","p_under"]
    show = recos.copy()
    if "confidence" not in show.columns and {"p_over","p_under","side"}.issubset(show.columns):
        show["confidence"] = np.where(show["side"].eq("Over"), show["p_over"], show["p_under"])

    for c in ["total_line","pred_total_mu","edge","kelly_frac","p_over","p_under","confidence"]:
        if c in show.columns:
            show[c] = show[c].astype(float).round(3)

    print("\nTOP PICKS (by edge):")
    print(show[cols].head(20).to_string(index=False))

# ---------- RUN ----------
diag = evaluate_model_cv(train_games, FEATURES, k=5, alpha=3.0, seed=42)
pretty_print_dict(diag, title="MODEL DIAGNOSTICS — CROSS-VALIDATION (HISTORICAL TOTALS)")
print_upcoming_summary(pred_df, recos, top_n=20)



MODEL DIAGNOSTICS — CROSS-VALIDATION (HISTORICAL TOTALS)
      CV folds: 5
    RMSE (pts): 13.14
     MAE (pts): 10.42
           R^2: 0.041
   Abs Err p50: 8.25
   Abs Err p75: 15.51
   Abs Err p90: 23.41

UPCOMING PICKS — SUMMARY
 Books x games: 13319
Positive-EV rows: 11005
Avg edge ($/1): 0.265
Avg confidence: 0.973
    Over picks: 12489
   Under picks: 830
    Sigma used: 13.2

Confidence buckets (count of picks):
  50–55%: 112
  55–60%: 65
  60–65%: 125
    65%+: 13017

TOP PICKS (by edge):
commence_time         book             home_key              away_key  total_line  pred_total_mu side  confidence  edge  kelly_frac  price_over  price_under  p_over  p_under
   10/26/2025 BetOnline.ag       denver broncos        dallas cowboys         2.5         59.160 Over         1.0 3.010        0.25       301.0         1.69     1.0      0.0
   10/26/2025 BetOnline.ag       denver broncos        dallas cowboys         2.5         59.160 Over         1.0 3.010        0.25       301.0      

In [2]:
import os, glob, time, requests, io, ssl, certifi, urllib.request
import pandas as pd, numpy as np
from datetime import datetime, timedelta
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from math import erf, sqrt

# -------------------------------------------------
# 1) Bring in your ESPN functions from the other notebook
#    (expects pull_players_and_teams_for_date to exist)
# -------------------------------------------------
%run ESPNDATA.ipynb

# --- Silence prints from ESPNDATA.ipynb helpers ---
from contextlib import redirect_stdout
import io
def pull_players_and_teams_for_date_quiet(ds):
    buf = io.StringIO()
    with redirect_stdout(buf):  # swallow ESPNDATA prints
        return pull_players_and_teams_for_date(ds)  # original function

# ---------- scalar-safe normalizer used everywhere ----------
def norm_key(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

# -------------------------------------------------
# 2) Utilities
# -------------------------------------------------
def payoff_from_american(odds):
    if pd.isna(odds):
        return np.nan
    o = int(round(float(odds)))
    return o/100.0 if o > 0 else 100.0/abs(o)

def normal_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def p_over_from_mu_sigma(line, mu, sigma):
    if sigma <= 1e-6:
        return float(mu > line)
    z = (line - mu) / sigma
    return 1 - normal_cdf(z)

# -------------------------------------------------
# 3) Ensure team points exist in teams_df (via ESPN summary header)
# -------------------------------------------------
SITE_SUMMARY = "https://site.web.api.espn.com/apis/site/v2/sports/football/nfl/summary"

def _j(url):
    r = requests.get(url, timeout=20); r.raise_for_status(); return r.json()

def _scores_for_event(event_id: str):
    h = _j(f"{SITE_SUMMARY}?event={event_id}").get("header", {})
    comp = (h.get("competitions") or [{}])[0]
    out = []
    for c in comp.get("competitors", []):
        team_name = c.get("team", {}).get("displayName")
        pts = c.get("score")
        if pts is None:
            ls = c.get("linescores") or []
            if ls:
                pts = sum(int(x.get("value") or 0) for x in ls)
        if team_name is not None and pts is not None:
            out.append({"event_id": str(event_id), "team": team_name, "points": int(pts)})
    return out

def ensure_points(teams_df: pd.DataFrame) -> pd.DataFrame:
    df = teams_df.copy()
    for candidate in ("points","pts","score"):
        if candidate in df.columns:
            return df.rename(columns={candidate: "points"}) if candidate != "points" else df

    score_rows = []
    for eid in df["event_id"].astype(str).unique():
        try:
            score_rows.extend(_scores_for_event(eid))
        except Exception:
            continue
        time.sleep(0.06)

    scores = pd.DataFrame(score_rows).dropna(subset=["points"])
    if scores.empty:
        raise ValueError("Couldn't derive team points; inspect teams_df and a sample summary JSON.")

    out = df.merge(scores, on=["event_id","team"], how="left")

    if out["points"].isna().any():
        df2 = out[out["points"].isna()].copy()
        ok  = out[out["points"].notna()]
        if not df2.empty:
            df2["team_key"] = df2["team"].map(norm_key)
            scores["team_key"] = scores["team"].map(norm_key)
            df2 = df2.drop(columns=["points"]).merge(
                scores.drop(columns=["team"]).drop_duplicates(["event_id","team_key"]),
                on=["event_id","team_key"], how="left"
            ).drop(columns=["team_key"])
            out = pd.concat([ok, df2], ignore_index=True)
    return out

# -------------------------------------------------
# 4) Build training set from ESPN team totals
# -------------------------------------------------
def pull_team_totals_for_dates(date_list):
    rows = []
    for ds in date_list:
        _, teams_df = pull_players_and_teams_for_date_quiet(ds)
        if teams_df.empty:
            continue
        rows.append(teams_df.assign(asof_date=ds))
        time.sleep(0.08)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def build_game_table_from_teams(teams_df):
    t = ensure_points(teams_df.copy())
    t['team_key'] = t['team'].map(norm_key)

    g = t[['event_id','team','team_key','points','asof_date']]
    g_sorted = g.sort_values(['event_id','team_key'])
    pairs = []
    for eid, grp in g_sorted.groupby('event_id'):
        if len(grp) != 2:
            continue
        a, b = grp.iloc[0], grp.iloc[1]
        pairs.append({
            'event_id': eid,
            'team_a': a.team, 'team_b': b.team,
            'team_a_key': a.team_key, 'team_b_key': b.team_key,
            'pts_a': pd.to_numeric(a.points, errors='coerce'),
            'pts_b': pd.to_numeric(b.points, errors='coerce'),
            'asof_date': a.asof_date
        })
    games = pd.DataFrame(pairs).dropna(subset=['pts_a','pts_b'])
    games['total_points'] = games['pts_a'] + games['pts_b']
    return games

def rolling_team_features(games, window=3):
    a = games[['event_id','asof_date','team_a_key','pts_a','pts_b']].rename(
        columns={'team_a_key':'team_key','pts_a':'pts_for','pts_b':'pts_against'})
    b = games[['event_id','asof_date','team_b_key','pts_b','pts_a']].rename(
        columns={'team_b_key':'team_key','pts_b':'pts_for','pts_a':'pts_against'})
    long = pd.concat([a,b], ignore_index=True).sort_values(['team_key','asof_date','event_id'])

    feats = []
    for team, grp in long.groupby('team_key'):
        grp = grp.copy()
        grp['pf_l3'] = grp['pts_for'].shift(1).rolling(window).mean()
        grp['pa_l3'] = grp['pts_against'].shift(1).rolling(window).mean()
        grp['pf_l5'] = grp['pts_for'].shift(1).rolling(5).mean()
        grp['pa_l5'] = grp['pts_against'].shift(1).rolling(5).mean()
        feats.append(grp.assign(team_key=team))
    f = pd.concat(feats, ignore_index=True)

    fa = f[['event_id','team_key','pf_l3','pa_l3','pf_l5','pa_l5']]
    fb = fa.copy()
    games2 = games.merge(
        fa, left_on=['event_id','team_a_key'], right_on=['event_id','team_key'], how='left'
    ).drop(columns=['team_key']).rename(columns={
        'pf_l3':'a_pf_l3','pa_l3':'a_pa_l3','pf_l5':'a_pf_l5','pa_l5':'a_pa_l5'
    })
    games2 = games2.merge(
        fb, left_on=['event_id','team_b_key'], right_on=['event_id','team_key'], how='left'
    ).drop(columns=['team_key']).rename(columns={
        'pf_l3':'b_pf_l3','pa_l3':'b_pa_l3','pf_l5':'b_pf_l5','pa_l5':'b_pa_l5'
    })
    return games2

def train_total_model(train_games):
    features = ['a_pf_l3','a_pa_l3','a_pf_l5','a_pa_l5','b_pf_l3','b_pa_l3','b_pf_l5','b_pa_l5']
    X = train_games[features].fillna(train_games[features].mean())
    y = train_games['total_points']
    model = Ridge(alpha=3.0).fit(X, y)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    preds, ys = [], []
    for tr, te in kf.split(X):
        m = Ridge(alpha=3.0).fit(X.iloc[tr], y.iloc[tr])
        p = m.predict(X.iloc[te])
        preds.append(p); ys.append(y.iloc[te].values)
    resid = np.concatenate(ys) - np.concatenate(preds)
    sigma = np.std(resid, ddof=1)
    return model, sigma, features

def build_training_from_dates(n_days=200):
    today = datetime.utcnow().date()
    dates = [(today - timedelta(days=i)).strftime("%Y%m%d") for i in range(n_days)]
    ttot = pull_team_totals_for_dates(dates)
    games = build_game_table_from_teams(ttot)
    games = rolling_team_features(games)
    games = games.dropna(subset=['a_pf_l3','b_pf_l3'])
    return games

# -------------------------------------------------
# 5) READ GAME TOTALS (Over/Under) FROM GOOGLE SHEETS — EXPANDED SYNONYMS
# -------------------------------------------------
SHEET_CSV_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQtfhqFKMwFDldCgWJp4Lb5wqm71F2EXUdwYD_75VxAMPlyUsoMaWct5KrYwXJyPScMxTKLjonLrEbB/pub?gid=0&single=true&output=csv"

def read_totals_from_google_sheet(csv_url: str) -> pd.DataFrame:
    """Return one row per game+book with book, home_team_api, away_team_api, point,
       commence_time, price_over, price_under, home_key, away_key.
       Supports wide (price_over/price_under) or long (Over/Under rows)."""
    # Robust CSV fetch (handles SSL constraints)
    try:
        ctx = ssl.create_default_context(cafile=certifi.where())
        with urllib.request.urlopen(csv_url, context=ctx, timeout=30) as resp:
            raw = pd.read_csv(io.BytesIO(resp.read()))
    except Exception:
        r = requests.get(csv_url, timeout=30)
        r.raise_for_status()
        raw = pd.read_csv(io.BytesIO(r.content))

    def _pick_col(df, candidates):
        cmap = {c.lower(): c for c in df.columns}
        for c in candidates:
            if c in df.columns:
                return c
            if c.lower() in cmap:
                return cmap[c.lower()]
        return None

    # ----- find key columns (expanded synonyms) -----
    home_col  = _pick_col(raw, ["home_team_api","home_team","home"])
    away_col  = _pick_col(raw, ["away_team_api","away_team","away"])
    book_col  = _pick_col(raw, ["book","bookmaker"])
    point_col = _pick_col(raw, ["point","total","line","total_points",
                                "over_under","ou_total","ou","total_line","totals"])
    time_col   = _pick_col(raw, ["commence_time","start_time","kickoff","starts_at"])
    market_col = _pick_col(raw, ["market","bet_type"])
    name_col   = _pick_col(raw, ["name","label"])  # long format
    price_col  = _pick_col(raw, ["price","odds","american_odds"])
    over_col   = _pick_col(raw, ["price_over","over_price","overodds","over_odds","over"])
    under_col  = _pick_col(raw, ["price_under","under_price","underodds","under_odds","under"])

    if home_col is None or away_col is None or book_col is None or point_col is None:
        raise ValueError(f"Missing core totals columns. Found: {list(raw.columns)}")

    # quick debug mapping
    print(f"[sheet map] book={book_col} home={home_col} away={away_col} line={point_col} "
          f"time={time_col or 'N/A'} over={over_col or 'N/A'} under={under_col or 'N/A'}")

    df = raw.copy()

    # Enforce totals-only, if a market column exists
    if market_col and market_col in df.columns:
        mask = df[market_col].astype(str).str.lower().eq("totals")
        if mask.any():
            df = df[mask].copy()
        else:
            raise ValueError("Sheet market is not 'totals'. Use a totals-only tab for O/U.")

    # ========== Wide format ==========
    if over_col and under_col:
        use_cols = [book_col, home_col, away_col, point_col, over_col, under_col] + ([time_col] if time_col else [])
        w = df[use_cols].rename(columns={
            book_col: "book",
            home_col: "home_team_api",
            away_col: "away_team_api",
            point_col: "point",
            (time_col or "commence_time"): "commence_time",
            over_col: "price_over",
            under_col: "price_under",
        })
    else:
        # ========== Long format (Over/Under rows) ==========
        if not (name_col and price_col and point_col):
            raise ValueError("Totals tab must include Over/Under prices (wide or long).")
        on_cols = [book_col, home_col, away_col, point_col] + ([time_col] if time_col else [])
        over  = df[df[name_col].astype(str).str.lower().eq("over") ][on_cols + [price_col]].copy()
        under = df[df[name_col].astype(str).str.lower().eq("under")][on_cols + [price_col]].copy()
        w = over.merge(under, on=on_cols, suffixes=("_over","_under")).rename(columns={
            book_col: "book",
            home_col: "home_team_api",
            away_col: "away_team_api",
            point_col: "point",
            (time_col or "commence_time"): "commence_time",
            f"{price_col}_over":  "price_over",
            f"{price_col}_under": "price_under",
        })

    # Clean + sanity guard
    w["price_over"]  = pd.to_numeric(w.get("price_over"),  errors="coerce")
    w["price_under"] = pd.to_numeric(w.get("price_under"), errors="coerce")
    w["point"]       = pd.to_numeric(w.get("point"),       errors="coerce")
    w = w.dropna(subset=["price_over","price_under","point"])
    if not ((w["point"] >= 20) & (w["point"] <= 80)).any():
        raise ValueError("Lines are not in a typical NFL totals range (20–80). Check that this tab is O/U only.")

    w["home_key"] = w["home_team_api"].map(norm_key)
    w["away_key"] = w["away_team_api"].map(norm_key)
    return w

# =================================================
# ===== 1) TRAIN
# =================================================
train_games = build_training_from_dates(n_days=200)
model, sigma, FEATURES = train_total_model(train_games)

# ---- Print training metrics only (no noisy dataframes) ----
def evaluate_model_cv(train_games, features, alpha=3.0, k=5, seed=42):
    X = train_games[features].fillna(train_games[features].mean())
    y = train_games['total_points'].values
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)

    preds, actuals = [], []
    for tr, te in kf.split(X):
        m = Ridge(alpha=alpha).fit(X.iloc[tr], y[tr])
        preds.append(m.predict(X.iloc[te]))
        actuals.append(y[te])

    p_all = np.concatenate(preds)
    y_all = np.concatenate(actuals)

    mse = mean_squared_error(y_all, p_all)   # compute RMSE manually for old sklearn
    rmse = float(np.sqrt(mse))

    return {
        "rmse": rmse,
        "mae" : mean_absolute_error(y_all, p_all),
        "r2"  : r2_score(y_all, p_all),
        "sigma_cv": float(np.std(y_all - p_all, ddof=1)),
        "n_games": int(len(y_all))
    }

def print_training_summary(metrics, title="MODEL (Totals) — Cross-Validated Metrics"):
    print("\n" + title)
    print("-"*len(title))
    print(f"N games : {metrics['n_games']}")
    print(f"RMSE    : {metrics['rmse']:.2f} pts")
    print(f"MAE     : {metrics['mae']:.2f} pts")
    print(f"R²      : {metrics['r2']:.3f}")
    print(f"σ (CV)  : {metrics['sigma_cv']:.2f} pts\n")

metrics = evaluate_model_cv(train_games, FEATURES, alpha=3.0, k=5, seed=42)
print_training_summary(metrics)

# =================================================
# ===== 2) READ ODDS FROM GOOGLE SHEETS (TOTALS ONLY)
# =================================================
odds = read_totals_from_google_sheet(SHEET_CSV_URL)
if odds.empty:
    raise RuntimeError("Odds sheet returned 0 rows. Double-check the sheet/tab & column names.")

# =================================================
# ===== 3) SCORE VS ODDS 
# =================================================
latest = train_games.copy()

def get_latest_feats(team_key, side_prefix='a'):
    cols = [f'{side_prefix}_pf_l3', f'{side_prefix}_pa_l3', f'{side_prefix}_pf_l5', f'{side_prefix}_pa_l5']
    a = latest[latest['team_a_key']==team_key][['event_id','team_a_key']+cols].tail(1)
    if not a.empty:
        return a.iloc[0][cols].values
    cols_b = ['b_pf_l3','b_pa_l3','b_pf_l5','b_pa_l5']
    b = latest[latest['team_b_key']==team_key][['event_id','team_b_key']+cols_b].tail(1)
    if not b.empty:
        return b.iloc[0][cols_b].values
    return [np.nan, np.nan, np.nan, np.nan]

# Build feature rows per upcoming game/book from the sheet
pred_rows = []
for _, r in odds.iterrows():
    home_k = norm_key(r['home_team_api'])
    away_k = norm_key(r['away_team_api'])
    a_feats = get_latest_feats(home_k, 'a')
    b_feats = get_latest_feats(away_k, 'b')
    pred_rows.append({
        'home_key': home_k, 'away_key': away_k,
        'book': r['book'], 'total_line': r['point'],
        'price_over': r['price_over'], 'price_under': r['price_under'],
        'commence_time': r['commence_time'],
        'a_pf_l3': a_feats[0], 'a_pa_l3': a_feats[1], 'a_pf_l5': a_feats[2], 'a_pa_l5': a_feats[3],
        'b_pf_l3': b_feats[0], 'b_pa_l3': b_feats[1], 'b_pf_l5': b_feats[2], 'b_pa_l5': b_feats[3],
    })

pred_df = pd.DataFrame(pred_rows).dropna(subset=["total_line", "price_over", "price_under"])

# Predict totals and uncertainty
X_new = pred_df[FEATURES].fillna(train_games[FEATURES].mean())
pred_df['pred_total_mu'] = model.predict(X_new)
pred_df['pred_sigma'] = sigma

# Probabilities & EVs
pred_df['p_over']  = pred_df.apply(lambda r: p_over_from_mu_sigma(r['total_line'], r['pred_total_mu'], r['pred_sigma']), axis=1)
pred_df['p_under'] = 1 - pred_df['p_over']
pred_df['payoff_over']  = pred_df['price_over'].map(payoff_from_american)
pred_df['payoff_under'] = pred_df['price_under'].map(payoff_from_american)
pred_df['ev_over']  = pred_df['p_over']  * pred_df['payoff_over']  - (1 - pred_df['p_over'])
pred_df['ev_under'] = pred_df['p_under'] * pred_df['payoff_under'] - (1 - pred_df['p_under'])

# Pick side, edge, confidence
pred_df['side']       = np.where(pred_df['ev_over'] >= pred_df['ev_under'], 'Over', 'Under')
pred_df['edge']       = pred_df[['ev_over','ev_under']].max(axis=1)
pred_df['confidence'] = np.where(pred_df['side']=='Over', pred_df['p_over'], pred_df['p_under'])

# Kelly sizing (quarter Kelly)
def kelly_fraction(p, odds_american, q=0.25):
    if pd.isna(p) or pd.isna(odds_american): return 0.0
    b = payoff_from_american(odds_american)
    if pd.isna(b) or b <= 0: return 0.0
    f = (p*(b+1)-1)/b
    return max(0.0, q*float(f))

pred_df['kelly_frac'] = np.where(
    pred_df['side']=='Over',
    pred_df.apply(lambda r: kelly_fraction(r['p_over'],  r['price_over']),  axis=1),
    pred_df.apply(lambda r: kelly_fraction(r['p_under'], r['price_under']), axis=1)
)

# Columns to show
cols = ['commence_time','book','home_key','away_key','total_line',
        'pred_total_mu','pred_sigma','side','confidence','edge','kelly_frac',
        'price_over','price_under','p_over','p_under']

# Build the sorted recommendations table
recos = pred_df[cols].sort_values('edge', ascending=False).reset_index(drop=True)

# --- Pretty, labeled table of the top N picks (with headers) ---
def show_top_picks(recos_df, top_n=20):
    if recos_df is None or recos_df.empty:
        print("No recommendations to show.")
        return

    cols_pretty = [
        "commence_time","book","home_key","away_key","total_line",
        "pred_total_mu","side","confidence","edge","kelly_frac",
        "price_over","price_under","p_over","p_under"
    ]
    rename_map = {
        "commence_time":"Date/Time",
        "book":"Book",
        "home_key":"Home",
        "away_key":"Away",
        "total_line":"O/U Line",
        "pred_total_mu":"Model Total",
        "side":"Pick",
        "confidence":"Conf",
        "edge":"Edge($/1)",
        "kelly_frac":"Kelly f",
        "price_over":"Over odds",
        "price_under":"Under odds",
        "p_over":"P(Over)",
        "p_under":"P(Under)"
    }

    t = recos_df.loc[:, cols_pretty].rename(columns=rename_map).copy()
    for c in ["O/U Line","Model Total","Conf","Edge($/1)","Kelly f",
              "Over odds","Under odds","P(Over)","P(Under)"]:
        if c in t.columns:
            t[c] = pd.to_numeric(t[c], errors="coerce").round(3)

    pd.set_option("display.width", 180)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_colwidth", 40)

    print("\n=== TOP PICKS (by edge) ===")
    print(t.head(top_n).to_string(index=False, header=True))

# Show the top 20 in a clear, labeled table
show_top_picks(recos, top_n=20)

# Optional: also save to CSV for your app
recos.to_csv("ou_recommendations_upcoming.csv", index=False)


Found 16 games via: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?dates=2024&seasontype=2&week=1
Rows: 1228
 season  week             team stat_type           player athlete_id  seasontype  event_id C/ATT YDS  AVG  TD INT SACKS
   2024     1 Baltimore Ravens   passing    Lamar Jackson    3916387           2 401671789 26/41 273  6.7   1   0   1-6
   2024     1 Baltimore Ravens   rushing    Lamar Jackson    3916387           2 401671789   NaN 122  7.6   0 NaN   NaN
   2024     1 Baltimore Ravens   rushing    Derrick Henry    3043078           2 401671789   NaN  46  3.5   1 NaN   NaN
   2024     1 Baltimore Ravens   rushing      Zay Flowers    4429615           2 401671789   NaN  14  7.0   0 NaN   NaN
   2024     1 Baltimore Ravens   rushing     Justice Hill    4038441           2 401671789   NaN   3  3.0   0 NaN   NaN
   2024     1 Baltimore Ravens receiving    Isaiah Likely    4361050           2 401671789   NaN 111 12.3   1 NaN   NaN
   2024     1 Baltimore Rave

/var/folders/pj/1wf8h2rx47nf6pwhy55jvs2c0000gn/T/ipykernel_90425/2759421428.py:187: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow().date()



MODEL (Totals) — Cross-Validated Metrics
----------------------------------------
N games : 135
RMSE    : 13.59 pts
MAE     : 10.51 pts
R²      : -0.016
σ (CV)  : 13.64 pts



ValueError: Missing core totals columns. Found: ['game_id', 'commence_time', 'in_play', 'bookmaker', 'last_update', 'home_team', 'away_team', 'market', 'label_1', 'odd_1', 'point_1', 'label_2', 'odd_2', 'point_2', 'odd_draw']